In [1]:
import warnings
# Ignore all warnings
warnings.filterwarnings('ignore')

# Enable interactive mode with a scrollable plot
%matplotlib qt
import numpy as np
import glob
import sys
import pickle as pkl
sys.path.append("..")

import mne
import gc

from matplotlib.colors import ListedColormap
import socket
import pickle as pkl
from pathlib import Path

%load_ext autoreload
%autoreload 2

In [2]:
import warnings
warnings.filterwarnings('ignore')

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
    "gamma high": (45, 80)
}

bands_stats = {k: v for k, v in bands.items() if k != "gamma high"}

paradigms = {0: "noise", 1: "music"}
pat_state = {"sub01": "UWS", "sub03": "MCS+", "sub04": "MCS-", "sub07":"MCS-", "sub09":"MCS-", "sub10":"MCS-", "sub12":"MCS-"}
cmap_color = ListedColormap(['black', 'white'])

In [3]:
device_name = socket.gethostname()
print(device_name)

DESKTOP-I0FNO1F


In [4]:
if device_name == "LAPTOP-LRI6535I":
    path_analysis_data_new = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "E:/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/"

elif device_name == "DESKTOP-I0FNO1F":
    path_analysis_data = "D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/"

elif device_name == "kritrim":
    path_analysis_data = "C:/Users/manda/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"

else:
    print("Define the Correct paths")



In [5]:
from helpers.frequency_analysis import compute_psd_mne
from helpers.helper_functions import get_fooof_bands, get_fooof_aperiodic, get_matched_legths, get_fooof

from helpers.topo_eeg import plot_topomap_, get_fdr_p_values, plot_topomap_opt, plot_band_power_difference_all_subjects_significance_aperiodic
from helpers.topo_eeg import plot_band_power_difference_all_subjects, get_channel_wise_fooof, plot_aperiodic_difference_all_subjects

from helpers.extract_data import crop_to_fixed_length, remove_all_annotations
from helpers.group_level_helpers import group_level_bootstrap_test, latex_group_vs_individual_table


def store_mne_raw(eeg_filtered, path_folder, path_data, ovwrite=False):
    eeg_filtered.save(path_folder + path_data, overwrite=ovwrite)

def load_mne_raw(path_folder, path_data):
    return mne.io.read_raw_fif(path_folder + path_data, preload=True)


Python Executable Path: 
c:\Users\specsdcn\anaconda3\envs\pdoc\python.exe


d:\Taran\OneDrive - Radboud Universiteit\Documents\music_doc_study\analysis\..\helpers\helper_functions.py:9: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


Load All Data

In [6]:
participants = ["p1", "p3", "p4", "p7"]
eeg_data_all = {pt:[] for pt in participants}
eeg_data_all


for pt in participants:
    path_folder = path_analysis_data + pt + "/"
    cleaned_data_file = glob.glob(f"{path_folder}/*_cleaned.fif")
    print(cleaned_data_file)

    for fls in cleaned_data_file:
        cleaned_data_file_name = fls.split("\\")[1]
        parad = cleaned_data_file_name.split("_")[0]
        loaded_file_name = f"{pt}_{parad}_eeg_cleaned" 
        print(loaded_file_name)
        print("Paradigm:", parad)
        
        if parad == "rest":
            data_file_rest = load_mne_raw(path_folder, cleaned_data_file_name)

        elif parad == "passive" or parad == "reward":
            data_file_passive = load_mne_raw(path_folder, cleaned_data_file_name)
    
    len_rest = float(data_file_rest.times[-1])
    len_passive = float(data_file_passive.times[-1])

    print("\n\n\n")
    print(f"Length of rest data: {len_rest:.2f}s")
    print(f"Length of passive data: {len_passive:.2f}s")


    if len_rest > len_passive:
        data_file_rest = crop_to_fixed_length(data_file_rest, len_passive)

    elif len_passive > len_rest:
        data_file_passive = crop_to_fixed_length(data_file_passive, len_rest)

    len_rest_new = float(data_file_rest.times[-1])
    len_passive_new = float(data_file_passive.times[-1])

    print(f"Length of rest data new: {len_rest_new:.2f}s")
    print(f"Length of passive data new: {len_passive_new:.2f}s")

    print("\n\n\n")

    passive_cleaned_npy = data_file_passive.get_data()
    rest_cleaned_npy = data_file_rest.get_data()

    eeg_data_all[pt].append(remove_all_annotations(data_file_rest.copy()))
    eeg_data_all[pt].append(remove_all_annotations(data_file_passive.copy()))

    del data_file_rest
    del data_file_passive
    gc.collect()

['D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p1\\passive_ica_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p1\\rest_ica_cleaned.fif']
p1_passive_eeg_cleaned
Paradigm: passive
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p1/passive_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p1/passive_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 0 ... 220529 =      0.000 ...   430.721 secs
Ready.
Reading 0 ... 220529  =      0.000 ...   430.721 secs...
p1_rest_eeg_cleaned
Paradigm: rest
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p1/rest_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p1/rest_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 0 ... 205527 =      0.000 ...   401.420 secs
Ready.
Reading 0 ... 205527  =      0.000 ...   401.420 secs...




Length of rest data: 401.42s
Length of passive data: 430.72s
Cropped → from beginning → 0.00s to 401.42s (duration = 401.418s)
Length of rest data new: 401.42s
Length of passive data new: 401.42s




['D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p3\\passive_ica_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p3\\rest_ica_cleaned.fif']
p3_passive_eeg_cleaned
Paradigm: passive
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p3/passive_ica_cleaned.fif...


...

    Range : 0 ... 91626 =      0.000 ...   178.957 secs
Ready.
Reading 0 ... 91626  =      0.000 ...   178.957 secs...




Length of rest data: 178.96s
Length of passive data: 191.74s
Cropped → from beginning → 0.00s to 178.96s (duration = 178.955s)
Length of rest data new: 178.96s
Length of passive data new: 178.96s




['D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7\\eeg_music_ica_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7\\eeg_pink_ica_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7\\eeg_rest_end_ica_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7\\eeg_rest_mid_ica_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7\\music_segs_cleaned.fif', 'D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analy

C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/rest_ica_init_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 0 ... 198434 =      0.000 ...   387.566 secs
Ready.
Reading 0 ... 198434  =      0.000 ...   387.566 secs...
p7_reward_eeg_cleaned
Paradigm: reward
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/reward_after_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/reward_after_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 0 ... 188386 =      0.000 ...   367.941 secs
Ready.
Reading 0 ... 188386  =      0.000 ...   367.941 secs...




Length of rest data: 387.57s
Length of passive data: 367.94s
Cropped → from beginning → 0.00s to 367.94s (duration = 367.939s)
Length of rest data new: 367.94s
Length of passive data new: 367.94s






2. Import Data from Sub07

In [7]:
music_segs_cleaned_file = "p7/music_segs_cleaned_trimmed.fif"
music_segs_cleaned_trimmed_sub07 = load_mne_raw(path_analysis_data, music_segs_cleaned_file)

pink_segs_cleaned_file = "p7/pink_segs_cleaned_trimmed.fif"
pink_segs_cleaned_trimmed_sub07 = load_mne_raw(path_analysis_data, pink_segs_cleaned_file)

rest_mid_cleaned_file = "p7/eeg_rest_mid_ica_cleaned.fif"
rest_mid_cleaned_trimmed_sub07 = load_mne_raw(path_analysis_data, rest_mid_cleaned_file)

rest_end_cleaned_file = "p7/eeg_rest_end_ica_cleaned.fif"
rest_end_cleaned_trimmed_sub07 = load_mne_raw(path_analysis_data, rest_end_cleaned_file)

Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/music_segs_cleaned_trimmed.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/music_segs_cleaned_trimmed.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 512 ... 321472 =      1.000 ...   627.875 secs
Ready.
Reading 0 ... 320960  =      0.000 ...   626.875 secs...
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/pink_segs_cleaned_trimmed.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/pink_segs_cleaned_trimmed.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 0 ... 328790 =      0.000 ...   642.168 secs
Ready.
Reading 0 ... 328790  =      0.000 ...   642.168 secs...
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/eeg_rest_mid_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/eeg_rest_mid_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 1024 ... 95177 =      2.000 ...   185.893 secs
Ready.
Reading 0 ... 94153  =      0.000 ...   183.893 secs...
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/eeg_rest_end_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/p7/eeg_rest_end_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 1024 ... 96476 =      2.000 ...   188.430 secs
Ready.
Reading 0 ... 95452  =      0.000 ...   186.430 secs...


3. Import Data from Sub10

In [8]:
def remove_bad_edge_annotations(raw, duration=0.5):
    onset_list = []
    duration_list = []
    description_list = []
    raw_annots = raw.annotations

    for ann in raw_annots:
        if "EDGE" not in ann["description"] and "BAD"  not in ann["description"]:
            onset_list.append(ann["onset"])
            duration_list.append(duration)
            description_list.append(ann["description"])


    annots = mne.annotations.Annotations(
        onset=onset_list,
        duration=duration_list,
        description=description_list)

    return annots


In [9]:
subs_to_load = {"sub09", "sub10", "sub12"}
music_segs_cleaned_file = "eeg_all_ica_cleaned.fif"
data_cleaned_all = {}   


for sub in subs_to_load:
    raw_sub = load_mne_raw(path_analysis_data_new, sub + "/" + music_segs_cleaned_file)
    sub_anns = remove_bad_edge_annotations(raw_sub)
    raw_sub.set_annotations(sub_anns)
    data_cleaned_all[sub] = raw_sub
    

Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/sub09/eeg_all_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/sub09/eeg_all_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Read a total of 1 projection items:
        Average EEG reference (1 x 128)  idle
    Range : 0 ... 1236775 =      0.000 ...  2415.576 secs
Ready.
Reading 0 ... 1236775  =      0.000 ...  2415.576 secs...
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/sub10/eeg_all_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\4251347852.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw_sub.set_annotations(sub_anns)
C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/sub10/eeg_all_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Range : 0 ... 656133 =      0.000 ...  1281.510 secs
Ready.
Reading 0 ... 656133  =      0.000 ...  1281.510 secs...
Opening raw data file D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/sub12/eeg_all_ica_cleaned.fif...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\4251347852.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw_sub.set_annotations(sub_anns)
C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3544818026.py:15: RuntimeWarning: This filename (D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/sub12/eeg_all_ica_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return mne.io.read_raw_fif(path_folder + path_data, preload=True)


    Read a total of 1 projection items:
        Average EEG reference (1 x 128)  idle
    Range : 0 ... 1088720 =      0.000 ...  2126.406 secs
Ready.
Reading 0 ... 1088720  =      0.000 ...  2126.406 secs...


C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\4251347852.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw_sub.set_annotations(sub_anns)


In [10]:
for sub in data_cleaned_all:
    all_anns = data_cleaned_all[sub].annotations
    print(sub, len(all_anns))


sub09 45
sub10 44
sub12 45


In [11]:
for ann in data_cleaned_all["sub12"].annotations:
    print(ann["description"], ann["onset"])

rest_init_end 353.5
Boundary_1_Kabhi_Khushi_Kabhie_Gham_283.8 375.55426
Boundary_2_pink_noise_audio_180.0 397.860413
Boundary_3_Rahein_Na_Rahein_Hum_90.0 420.280396
Boundary_4_pink_noise_audio_540.0 440.009949
Boundary_5_Kabhi_Khushi_Kabhie_Gham_270.0 460.746277
Boundary_6_pink_noise_audio_360.0 481.429443
Boundary_7_Kabhi_Khushi_Kabhie_Gham_240.0 502.501221
Boundary_8_pink_noise_audio_30.0 524.431458
Boundary_9_Rahein_Na_Rahein_Hum_150.0 546.403687
Boundary_10_pink_noise_audio_300.0 568.572144
Boundary_11_Kabhi_Khushi_Kabhie_Gham_210.0 589.643982
Boundary_12_pink_noise_audio_450.0 609.356934
Boundary_13_Kabhi_Khushi_Kabhie_Gham_60.0 630.202637
Boundary_14_pink_noise_audio_390.0 654.345764
Boundary_15_Kabhi_Khushi_Kabhie_Gham_180.0 676.210083
Boundary_16_pink_noise_audio_150.0 696.539185
Boundary_17_Rahein_Na_Rahein_Hum_180.0 714.648621
Boundary_18_pink_noise_audio_330.0 733.669617
Boundary_19_Rahein_Na_Rahein_Hum_330.0 753.079346
Boundary_20_pink_noise_audio_90.0 772.484619
rest_mid_e

In [12]:
def reset_annotations(raw, paradigm):
    onset_list = []
    duration_list = []
    description_list = []

    duration = 0.5
    
    onset_start = 0.0
    onset_end = raw.times[-1]

    onset_list = [onset_start, onset_end]
    duration_list = [duration, duration]

    description_start = paradigm + "_start"
    description_end = paradigm + "_end"
    description_list = [description_start, description_end]

    annots = mne.annotations.Annotations(
        onset=onset_list,
        duration=duration_list,
        description=description_list)

    return annots

In [13]:
segmented_data_all = {sub: {} for sub in subs_to_load}

for sub in subs_to_load:    
    t_start = 0
    t_end = 0

    # t_start_segs = 0
    # t_end_segs = 0

    count_ann = 0
    all_annots = data_cleaned_all[sub].annotations

    offset = data_cleaned_all[sub].first_time

    if offset < 0:
        raise ValueError("Offset is negative!")

    segmented_data_all[sub]["pink_segs"] = []
    segmented_data_all[sub]["music_segs"] = []

    for ann in all_annots:
        t_end = ann["onset"] - offset
        seg = data_cleaned_all[sub].copy().crop(tmin=t_start, tmax=t_end, include_tmax=False)
        paradigm_name = ann["description"]
        annots_reset = reset_annotations(seg, paradigm_name)
        seg.set_annotations(annots_reset)

        # Handle the segmented passive data:

        if "end" not in paradigm_name:
            if "pink" in paradigm_name:
                segmented_data_all[sub]["pink_segs"].append(seg)
            
            else:
                segmented_data_all[sub]["music_segs"].append(seg)

        else:
            paradigm_name_split = paradigm_name.split("_")
            paradigm_name_shortened = paradigm_name_split[0] + "_" + paradigm_name_split[1]
            segmented_data_all[sub][paradigm_name_shortened] = seg


        count_ann += 1
        t_start = t_end
    
    segmented_data_all[sub]["pink_segs"] = mne.concatenate_raws(segmented_data_all[sub]["pink_segs"])
    segmented_data_all[sub]["music_segs"] = mne.concatenate_raws(segmented_data_all[sub]["music_segs"])

C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3479010342.py:26: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  seg.set_annotations(annots_reset)
C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3479010342.py:26: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  seg.set_annotations(annots_reset)
C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3479010342.py:26: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  seg.set_annotations(annots_reset)
C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3479010342.py:26: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  seg.set_annotations(annots_reset)
C:\Users\specsdcn\AppData\Local\Temp\ipykernel_25228\3479010342.py:26: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  seg.set_annotations(annots_reset)
C:\Users\specsdcn\AppData\Local\Temp\ipykerne

In [14]:
for sub in segmented_data_all:
    for parad in segmented_data_all[sub]:
        raw = segmented_data_all[sub][parad]
        annots = raw.annotations
        print()
        print("Paradigm Name: ", parad, "Raw First time: ", raw.first_time)
        for ann in annots:
            print(ann["onset"], ann["description"])


Paradigm Name:  pink_segs Raw First time:  467.169921875
467.169921875 Boundary_2_pink_noise_audio_150.0_start
490.40039087499997 Boundary_2_pink_noise_audio_150.0_end
490.40234375 BAD boundary
490.40234375 EDGE boundary
490.402343875 Boundary_4_pink_noise_audio_420.0_start
510.699218875 Boundary_4_pink_noise_audio_420.0_end
510.701171875 BAD boundary
510.701171875 EDGE boundary
510.701171875 Boundary_6_pink_noise_audio_510.0_start
536.019530875 Boundary_6_pink_noise_audio_510.0_end
536.0214838750001 Boundary_8_pink_noise_audio_570.0_start
536.021484375 BAD boundary
536.021484375 EDGE boundary
560.460936875 Boundary_8_pink_noise_audio_570.0_end
560.462890625 BAD boundary
560.462890625 EDGE boundary
560.462890875 Boundary_10_pink_noise_audio_60.0_start
585.728515875 Boundary_10_pink_noise_audio_60.0_end
585.73046875 BAD boundary
585.73046875 EDGE boundary
585.730468875 Boundary_12_pink_noise_audio_300.0_start
611.269530875 Boundary_12_pink_noise_audio_300.0_end
611.2714838750001 Bounda

In [15]:
segmented_data_all_annots_cleaned = {}

for sub in segmented_data_all:
    segmented_data_all_annots_cleaned[sub] = {}

    for paradigm in segmented_data_all[sub]:
        onset_list = []
        description_list = []
        duration_list = []

        raw = segmented_data_all[sub][paradigm]
        raw_offset = raw.first_time

        if raw_offset < 0:
            raise ValueError("Offset is negative!")
        
        annots = raw.annotations

        for ann in annots:
            if "BAD" not in ann["description"] and "EDGE" not in ann["description"]:
                    onset_list.append(ann["onset"] - raw_offset)
                    description_list.append(ann["description"])
                    duration_list.append(ann["duration"])

        
        segmented_data_all_annots_cleaned[sub][paradigm] = raw.copy().set_annotations(
                                                                    mne.Annotations(onset=onset_list, 
                                                                    duration=duration_list, description=description_list))



In [16]:
for sub in segmented_data_all_annots_cleaned:
    for parad in segmented_data_all_annots_cleaned[sub]:
        raw = segmented_data_all_annots_cleaned[sub][parad]
        annots = raw.annotations
        print()
        print("Paradigm Name: ", parad, "   Raw First time: ", raw.first_time, "     Count Annotations: ", len(annots))
        print()
        for ann in annots:
            print(ann["onset"], ann["description"])


Paradigm Name:  pink_segs    Raw First time:  467.169921875      Count Annotations:  40

467.169921875 Boundary_2_pink_noise_audio_150.0_start
490.40039087499997 Boundary_2_pink_noise_audio_150.0_end
490.402343875 Boundary_4_pink_noise_audio_420.0_start
510.699218875 Boundary_4_pink_noise_audio_420.0_end
510.701171875 Boundary_6_pink_noise_audio_510.0_start
536.019530875 Boundary_6_pink_noise_audio_510.0_end
536.0214838750001 Boundary_8_pink_noise_audio_570.0_start
560.460936875 Boundary_8_pink_noise_audio_570.0_end
560.462890875 Boundary_10_pink_noise_audio_60.0_start
585.728515875 Boundary_10_pink_noise_audio_60.0_end
585.730468875 Boundary_12_pink_noise_audio_300.0_start
611.269530875 Boundary_12_pink_noise_audio_300.0_end
611.2714838750001 Boundary_14_pink_noise_audio_240.0_start
637.273436875 Boundary_14_pink_noise_audio_240.0_end
637.275390875 Boundary_16_pink_noise_audio_390.0_start
663.277343875 Boundary_16_pink_noise_audio_390.0_end
663.279296875 Boundary_18_pink_noise_audio_

In [17]:
def remove_all_annotations(raw):
    """
    Remove all annotations from an MNE Raw object.

    Args:
        raw (mne.io.Raw): Raw object with annotations.

    Returns:
        mne.io.Raw: Same Raw object with annotations removed.
    """
    raw_copy = raw.copy()
    raw_copy.set_annotations(mne.Annotations([], [], []))
    return raw_copy


segmented_data_all_annots_removed = {}
for sub in segmented_data_all_annots_cleaned:
    segmented_data_all_annots_removed[sub] = {}

    for parad in segmented_data_all_annots_cleaned[sub]:
        raw = segmented_data_all_annots_cleaned[sub][parad]
        segmented_data_all_annots_removed[sub][parad] = remove_all_annotations(raw)



for sub in segmented_data_all_annots_removed:
    for parad in segmented_data_all_annots_removed[sub]:
        raw = segmented_data_all_annots_removed[sub][parad]
        annots = raw.annotations
        print()
        print("Paradigm Name: ", parad, "   Raw First time: ", raw.first_time, "     Count Annotations: ", len(annots))
        print()
        for ann in annots:
            print(ann["onset"], ann["description"])


Paradigm Name:  pink_segs    Raw First time:  467.169921875      Count Annotations:  0


Paradigm Name:  music_segs    Raw First time:  441.158203125      Count Annotations:  0


Paradigm Name:  rest_init    Raw First time:  0.0      Count Annotations:  0


Paradigm Name:  rest_mid    Raw First time:  920.2109375      Count Annotations:  0


Paradigm Name:  rest_end    Raw First time:  1600.802734375      Count Annotations:  0


Paradigm Name:  reward_end    Raw First time:  1821.02734375      Count Annotations:  0


Paradigm Name:  pink_noise    Raw First time:  2159.96484375      Count Annotations:  0


Paradigm Name:  pink_segs    Raw First time:  251.490234375      Count Annotations:  0


Paradigm Name:  music_segs    Raw First time:  244.95703125      Count Annotations:  0


Paradigm Name:  rest_init    Raw First time:  0.0      Count Annotations:  0


Paradigm Name:  rest_mid    Raw First time:  472.791015625      Count Annotations:  0


Paradigm Name:  reward_end    Raw First t

#### Merge All Data

In [18]:
eeg_data_all.keys()

dict_keys(['p1', 'p3', 'p4', 'p7'])

In [19]:
new_sub_keys = {"p1":"sub01", "p3":"sub03", "p4":"sub04", "p7":"sub07"}

for sub in new_sub_keys:
    eeg_data_all[new_sub_keys[sub]] = eeg_data_all.pop(sub)

In [20]:
eeg_data_all.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07'])

In [21]:
eeg_data_all_subs = {}

for sub in eeg_data_all:
    eeg_data_all_subs[sub] = {}
    eeg_data_all_subs[sub]["rest_init"] = eeg_data_all[sub][0]
    eeg_data_all_subs[sub]["reward"] = eeg_data_all[sub][1]

In [22]:
eeg_data_all_subs["sub10"] = segmented_data_all_annots_removed["sub10"]
eeg_data_all_subs["sub10"]["reward"] = segmented_data_all_annots_removed["sub10"].pop("reward_end")

In [23]:
eeg_data_all_subs["sub07"]["pink_segs"] = pink_segs_cleaned_trimmed_sub07
eeg_data_all_subs["sub07"]["music_segs"] = music_segs_cleaned_trimmed_sub07
eeg_data_all_subs["sub07"]["rest_mid"] = rest_mid_cleaned_trimmed_sub07
eeg_data_all_subs["sub07"]["rest_end"] = rest_end_cleaned_trimmed_sub07


In [24]:
eeg_data_all_subs["sub12"] = segmented_data_all_annots_removed["sub12"]
eeg_data_all_subs["sub09"] = segmented_data_all_annots_removed["sub09"]


In [25]:
eeg_data_all_subs["sub12"]["reward"] = segmented_data_all_annots_removed["sub12"].pop("reward_end")
eeg_data_all_subs["sub09"]["reward"] = segmented_data_all_annots_removed["sub09"].pop("reward_end")


In [26]:
for sub in eeg_data_all_subs:
    paradigms = eeg_data_all_subs[sub].keys()
    print(f"Subject: {sub}, Paradigms: {paradigms}")
    for parad in paradigms:
        eeg_data_all_subs[sub][parad] = remove_all_annotations(eeg_data_all_subs[sub][parad])
        print(len(eeg_data_all_subs[sub][parad].annotations))

Subject: sub01, Paradigms: dict_keys(['rest_init', 'reward'])
0
0
Subject: sub03, Paradigms: dict_keys(['rest_init', 'reward'])
0
0
Subject: sub04, Paradigms: dict_keys(['rest_init', 'reward'])
0
0
Subject: sub07, Paradigms: dict_keys(['rest_init', 'reward', 'pink_segs', 'music_segs', 'rest_mid', 'rest_end'])
0
0
0
0
0
0
Subject: sub10, Paradigms: dict_keys(['pink_segs', 'music_segs', 'rest_init', 'rest_mid', 'pink_noise', 'reward'])
0
0
0
0
0
0
Subject: sub12, Paradigms: dict_keys(['pink_segs', 'music_segs', 'rest_init', 'rest_mid', 'rest_end', 'pink_noise', 'reward'])
0
0
0
0
0
0
0
Subject: sub09, Paradigms: dict_keys(['pink_segs', 'music_segs', 'rest_init', 'rest_mid', 'rest_end', 'pink_noise', 'reward'])
0
0
0
0
0
0
0


#### Get FOOOF

In [27]:

# sub_bands = ["sub09", "sub10",  "sub12"]

band_powers_all = {sub: {parad: [] for parad in eeg_data_all_subs[sub]} for sub in eeg_data_all_subs}
psds_all = {sub: {parad: [] for parad in eeg_data_all_subs[sub]} for sub in eeg_data_all_subs}

subjects = list(eeg_data_all_subs.keys())
paradigms = list(eeg_data_all_subs[subjects[0]].keys())
sfreq = eeg_data_all_subs[subjects[0]][paradigms[0]].info["sfreq"]


f_min = bands["delta"][0]
f_max = bands["gamma high"][1]

segment_parads = ['pink_segs', 'music_segs']
power_type = "relative"

for sub in eeg_data_all_subs:
    for parad in  eeg_data_all_subs[sub]:

        # if parad in segment_parads:
        #     segment_secs = 3

        # else:
        segment_secs = 5
        
        segment_size = int(sfreq * segment_secs)
        overlap = int(segment_size /2)

        band_powers, psds, freqs = compute_psd_mne(eeg_data_all_subs[sub][parad], bands, overlap, segment_size, f_min, f_max, power_type=power_type)

        band_powers_all[sub][parad] = band_powers
        psds_all[sub][parad] = [psds, freqs]

Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective window size : 5.000 (s)
Effective wind

In [28]:
psds_all["sub09"]["pink_segs"][0].shape

(128, 398, 188)

In [ ]:
# eeg_fooof_all = get_fooof(psds_all, freq_range=(0.5, 45), save_path=path_analysis_data_new + "fooof_all_new45/")

sub01 | rest_init | 128 channels x 159 segments
sub01 | reward | 128 channels x 159 segments
sub03 | rest_init | 128 channels x 153 segments
sub03 | reward | 128 channels x 153 segments
sub04 | rest_init | 128 channels x 70 segments
sub04 | reward | 128 channels x 70 segments
sub07 | rest_init | 128 channels x 146 segments
sub07 | reward | 128 channels x 146 segments
sub07 | pink_segs | 128 channels x 255 segments
sub07 | music_segs | 128 channels x 249 segments
sub07 | rest_mid | 128 channels x 72 segments
sub07 | rest_end | 128 channels x 73 segments
sub10 | pink_segs | 128 channels x 69 segments
sub10 | music_segs | 128 channels x 136 segments
sub10 | rest_init | 128 channels x 96 segments
sub10 | rest_mid | 128 channels x 46 segments
sub10 | pink_noise | 128 channels x 71 segments
sub10 | reward | 128 channels x 85 segments
sub12 | pink_segs | 128 channels x 166 segments
sub12 | music_segs | 128 channels x 164 segments
sub12 | rest_init | 128 channels x 140 segments
sub12 | rest_mi

1.2. Load FOOOF

In [29]:
files_fooof = glob.glob(path_analysis_data_new + "fooof_all_new45/*.pkl")
eeg_fooof_all_subs = {"sub01":{}, "sub03":{}, "sub04":{}, "sub07":{}, "sub09":{}, "sub10":{}, "sub12":{}}

for fl in files_fooof:
    file_stem = Path(fl).stem
    file_name = Path(fl).name

    sub_name = file_stem.split("_")[2][:5]
    parad_name = file_stem.split("_")[2][5:]

    try:
        parad_name_2 = file_stem.split("_")[3]
        parad_name += "_" + parad_name_2

    except:
        parad_name = parad_name

    print(sub_name, parad_name)
    
    with open(fl, "rb") as f:
        # buffered = io.BufferedReader(f, buffer_size=1024*1024*16)  # 16 MB buffer
        eeg_fooof_all_subs[sub_name][parad_name] = pkl.load(f)
        f.close()

sub01 rest_init
sub01 reward
sub03 rest_init
sub03 reward
sub04 rest_init
sub04 reward
sub07 music_segs
sub07 pink_segs
sub07 rest_end
sub07 rest_init
sub07 rest_mid
sub07 reward
sub09 music_segs
sub09 pink_noise
sub09 pink_segs
sub09 rest_end
sub09 rest_init
sub09 rest_mid
sub09 reward
sub10 music_segs
sub10 pink_noise
sub10 pink_segs
sub10 rest_init
sub10 rest_mid
sub10 reward
sub12 music_segs
sub12 pink_noise
sub12 pink_segs
sub12 rest_end
sub12 rest_init
sub12 rest_mid
sub12 reward


### 1. Rest vs. Reward

In [62]:
eeg_fooof_rest_reward = {}

eeg_fooof_rest_reward["sub01"] = {}
eeg_fooof_rest_reward["sub01"]["rest_init"] = eeg_fooof_all_subs["sub01"]["rest_init"].copy()
eeg_fooof_rest_reward["sub01"]["reward"] = eeg_fooof_all_subs["sub01"]["reward"].copy()

eeg_fooof_rest_reward["sub03"] = {}
eeg_fooof_rest_reward["sub03"]["rest_init"] = eeg_fooof_all_subs["sub03"]["rest_init"].copy()
eeg_fooof_rest_reward["sub03"]["reward"] = eeg_fooof_all_subs["sub03"]["reward"].copy()

eeg_fooof_rest_reward["sub04"] = {}
eeg_fooof_rest_reward["sub04"]["rest_init"] = eeg_fooof_all_subs["sub04"]["rest_init"].copy()
eeg_fooof_rest_reward["sub04"]["reward"] = eeg_fooof_all_subs["sub04"]["reward"].copy()

eeg_fooof_rest_reward["sub07"] = {}
eeg_fooof_rest_reward["sub07"]["rest_init"] = eeg_fooof_all_subs["sub07"]["rest_init"].copy()
eeg_fooof_rest_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_reward["sub09"] = {} 
eeg_fooof_rest_reward["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_reward["sub10"] = {} 
eeg_fooof_rest_reward["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_reward["sub12"] = {} 
eeg_fooof_rest_reward["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [63]:
eeg_fooof_rest_reward_matched = get_matched_legths(eeg_fooof_rest_reward)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 159 epochs
(128, 223, 159) (128, 223, 159)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 153 epochs
(128, 223, 153) (128, 223, 153)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 70 epochs
(128, 223, 70) (128, 223, 70)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 146 epochs
(128, 223, 146) (128, 223, 146)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 175 → 134 epochs (dropped last 41)
(128, 223, 134) (128, 223, 134)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 96 → 85 epochs (dropped last 11)
(128, 223, 85) (128, 223, 85)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 140 → 105 epochs (dropped last 35)
(128, 223, 105) (128, 223, 105)


In [64]:
bands_stats

{'delta': (0.5, 4),
 'theta': (4, 8),
 'alpha': (8, 12),
 'beta': (12, 30),
 'gamma low': (30, 45)}

In [65]:
import numpy as np
from matplotlib.colors import to_rgb, ListedColormap

def pencil(color, whiteness=0.45):
    """Blend a color toward white -> soft, hand-drawn feel."""
    c = np.array(to_rgb(color))
    return tuple(c + (1 - c) * whiteness)

# cmap_color = ListedColormap([pencil("forestgreen", 0.35), pencil("#DAE8EC", 0.5)])


In [66]:
sub_id = "sub10"

raw_info = eeg_data_all_subs[sub_id]["rest_init"].info
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]

rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/rest_init_music_cont_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"

_, rest_init_music_fooof_bands_pvalues_abs, fooof_rest_music = get_channel_wise_fooof(freqs, eeg_fooof_rest_reward_matched, bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(rest_init_music_fooof_bands_pvalues_abs)

# cmap_color = ListedColormap(["#D45E00", '#DAE8EC'])  # warm greige / muted amber

cmap_color = ListedColormap(["#9EC49C", "#F2F7F9"])
plot_topomap_(fdr_corrected_p_vals, raw_info, pat_state, bands_stats, rel_dif_fdr_fooof_fig_path, show_bands=True, 
              title="Rest vs. Music Cont.", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Paradigm 0 rest_init
Paradigm 1 reward
sub09
Paradigm 0 rest_init
Paradigm 1 reward
sub10
Paradigm 0 rest_init
Paradigm 1 reward
sub12
Paradigm 0 rest_init
Paradigm 1 reward
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


In [67]:
rest_init_music_fooof_bands_pvalues_abs.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07', 'sub09', 'sub10', 'sub12'])

In [68]:
mcs_minus_subjects = ["sub04", "sub07", "sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_rest_music,
    mcs_minus_subjects,
    condition1="rest_init",
    condition2="reward",   # confirm this matches your actual music-condition key
    bands=bands_stats,
    abs_rel=0,
    n_boot=1000,
)

for band in bands_stats:
    print(f"{band}: {sig_mask[band].sum()} / 128 channels significant at group level")


# First comparison — produces the full table opening
latex_code_1 = latex_group_vs_individual_table(
    fdr_corrected_p_vals, mcs_minus_subjects, sig_mask, bands,
    comparison_label="Rest vs. Continuous Music",
    mode='start',
)
print(latex_code_1)

delta: 1 / 128 channels significant at group level
theta: 0 / 128 channels significant at group level
alpha: 0 / 128 channels significant at group level
beta: 0 / 128 channels significant at group level
gamma low: 0 / 128 channels significant at group level
\begin{table}[htbp]
\centering
\caption{Group-level versus individual-level significance across channels and frequency bands.}
\label{tab:group_vs_individual}

\begin{tabularx}{\textwidth}{@{}l l X X X X@{}}
\toprule
\textbf{Comparison} &
\textbf{Band} &
\textbf{Group-level sig.} &
\textbf{Sig. in $\geq$1 subject} &
\textbf{Sig. in majority} &
\textbf{Sig. in all subjects} \\
\midrule

\multirow{5}{*}{\shortstack[l]{Rest vs.\\Continuous Music}}
& $\delta$ & 1/128 & 75/128 & 0/128 ($\geq$3/5) & 0/128 \\
& $\theta$ & 0/128 & 94/128 & 5/128 ($\geq$3/5) & 0/128 \\
& $\alpha$ & 0/128 & 117/128 & 29/128 ($\geq$3/5) & 0/128 \\
& $\beta$ & 0/128 & 126/128 & 13/128 ($\geq$3/5) & 0/128 \\
& $\gamma$ & 0/128 & 127/128 & 29/128 ($\geq$3/5) & 0/

In [69]:
fooof_rest_music["sub01"]["rest_init"][0]["delta"].shape

(128, 159)

In [70]:
save_plots_path = plots_path + "frequency/opt_plots/"
# Mode 4 — your exact request
plot_topomap_opt(
    fdr_corrected_p_vals, raw_info, pat_state, 
    bands_to_plot=["delta", "theta", "alpha", "beta", "gamma low"],
    save_dir=save_plots_path,
    filename_template="rest_music_cont_{band}.pdf",
    cmap=cmap_color
)
# produces: output_folder/rest_music_delta.pdf, rest_music_theta.pdf, rest_music_alpha.pdf


['delta', 'theta', 'alpha', 'beta', 'gamma low']
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/frequency/opt_plots/rest_music_cont_delta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/frequency/opt_plots/rest_music_cont_theta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/frequency/opt_plots/rest_music_cont_alpha.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/frequency/opt_plots/rest_music_cont_beta.pdf
Saved: D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/frequency/opt_plots/rest_music_cont_gamma low.pdf


In [71]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/rest_init_music_cont_abs.pdf"
_, eeg_fooof_bands_relative_rest_music = get_fooof_bands(eeg_fooof_rest_reward_matched, freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 

from matplotlib.colors import LinearSegmentedColormap

soft_div = LinearSegmentedColormap.from_list(
    "soft_div",
    [pencil("steelblue", 0.15),      # negative pole (lower in condition 2)
     "#F7F7F5",                      # near-white centre (≈ no difference)
     pencil("palevioletred", 0.2)])  # positive pole (higher in condition 2)



plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_rest_music, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=True, title="(Music Cont. - Rest)/Rest", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  

Aperiodic

In [72]:
eeg_fooof_rest_music_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_reward_matched)

plot_aperiodic_difference_all_subjects(
    eeg_fooof_rest_music_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/rest_init_music_cont_abs_aperiodic.pdf",
    show_params=True,
    title="(Music Cont. - Rest)/Rest",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True
)


Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_init', 'reward']
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward


In [73]:
aperiodic_plots_path = plots_path + "frequency/" + "/rest_init_music_cont_abs_aperiodic_fdr.pdf"
title_plot = "Music Cont. vs. Rest"
n_channels = 128

plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_rest_music_aperiodic, raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels,
                                                               title=title_plot, show_params=True,
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward


### 2. Rest vs. Noise Segs

In [74]:
eeg_fooof_rest_noise = {}

eeg_fooof_rest_noise["sub07"] = {} 
eeg_fooof_rest_noise["sub07"]["rest_init"] = eeg_fooof_all_subs["sub07"]["rest_init"].copy()
eeg_fooof_rest_noise["sub07"]["pink_segs"] = eeg_fooof_all_subs["sub07"]["pink_segs"].copy()

eeg_fooof_rest_noise["sub09"] = {} 
eeg_fooof_rest_noise["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_noise["sub09"]["pink_segs"]  = eeg_fooof_all_subs["sub09"]["pink_segs"].copy()

eeg_fooof_rest_noise["sub10"] = {} 
eeg_fooof_rest_noise["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_noise["sub10"]["pink_segs"] = eeg_fooof_all_subs["sub10"]["pink_segs"].copy()

eeg_fooof_rest_noise["sub12"] = {} 
eeg_fooof_rest_noise["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_noise["sub12"]["pink_segs"] = eeg_fooof_all_subs["sub12"]["pink_segs"].copy()

In [75]:
eeg_fooof_rest_pink_segs_matched = get_matched_legths(eeg_fooof_rest_noise)

Subject: sub07, Paradigms: ['rest_init', 'pink_segs']
Truncated array2 from 255 → 146 epochs (dropped last 109)
(128, 223, 146) (128, 223, 146)
Subject: sub09, Paradigms: ['rest_init', 'pink_segs']
Truncated array2 from 188 → 175 epochs (dropped last 13)
(128, 223, 175) (128, 223, 175)
Subject: sub10, Paradigms: ['rest_init', 'pink_segs']
Truncated array1 from 96 → 69 epochs (dropped last 27)
(128, 223, 69) (128, 223, 69)
Subject: sub12, Paradigms: ['rest_init', 'pink_segs']
Truncated array2 from 166 → 140 epochs (dropped last 26)
(128, 223, 140) (128, 223, 140)


In [76]:
fdr_corrected_p_vals.keys()

dict_keys(['sub01', 'sub03', 'sub04', 'sub07', 'sub09', 'sub10', 'sub12'])

In [77]:
pat_state

{'sub01': 'UWS',
 'sub03': 'MCS+',
 'sub04': 'MCS-',
 'sub07': 'MCS-',
 'sub09': 'MCS-',
 'sub10': 'MCS-',
 'sub12': 'MCS-'}

In [78]:
rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/rest_init_pink_segs_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]


_, rest_noise_fooof_bands_pvalues_abs, _ = get_channel_wise_fooof(freqs, eeg_fooof_rest_pink_segs_matched, bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(rest_noise_fooof_bands_pvalues_abs)
# x, y= get_channel_wise_fooof(band_powers_rest_music["psds"], eeg_fooof_all_subs, bands)

plot_topomap_(fdr_corrected_p_vals, raw_info, pat_state, bands, rel_dif_fdr_fooof_fig_path, show_bands=1, 
              title="Rest vs. Pink Noise Segs", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub07
Paradigm 0 rest_init
Paradigm 1 pink_segs
sub09
Paradigm 0 rest_init
Paradigm 1 pink_segs
sub10
Paradigm 0 rest_init
Paradigm 1 pink_segs
sub12
Paradigm 0 rest_init
Paradigm 1 pink_segs
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


In [79]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/rest_init_pink_segs_abs.pdf"

_, eeg_fooof_bands_relative_rest_noise = get_fooof_bands(eeg_fooof_rest_pink_segs_matched, freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 


plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_rest_noise, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=1, title="(Pink Noise Segs - Rest)/Rest", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names an

Aperiodic

In [80]:
eeg_fooof_rest_noise_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_pink_segs_matched)


plot_aperiodic_difference_all_subjects(
    eeg_fooof_rest_noise_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/rest_init_pink_segs_abs_aperiodic.pdf",
    show_params=True,
    title="(Pink Noise Segs - Rest)/Rest",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True
)


Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_segs']
Paradigm 0:  rest_init
Paradigm 1:  pink_segs
Paradigm 0:  rest_init
Paradigm 1:  pink_segs
Paradigm 0:  rest_init
Paradigm 1:  pink_segs
Paradigm 0:  rest_init
Paradigm 1:  pink_segs


In [81]:
aperiodic_plots_path = plots_path + "frequency/" + "/rest_init_pink_segs_abs_aperiodic_fdr.pdf"
title_plot = "Rest vs. Pink Noise Segs"

plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_rest_noise_aperiodic, raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels,
                                                               title=title_plot, show_params=True,
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  pink_segs
Paradigm 0:  rest_init
Paradigm 1:  pink_segs
Paradigm 0:  rest_init
Paradigm 1:  pink_segs
Paradigm 0:  rest_init
Paradigm 1:  pink_segs


### 3. Rest vs. Noise Cont

In [82]:
eeg_fooof_rest_noise_cont = {}

eeg_fooof_rest_noise_cont["sub09"] = {} 
eeg_fooof_rest_noise_cont["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_noise_cont["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()


eeg_fooof_rest_noise_cont["sub10"] = {} 
eeg_fooof_rest_noise_cont["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_noise_cont["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()

eeg_fooof_rest_noise_cont["sub12"] = {} 
eeg_fooof_rest_noise_cont["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_noise_cont["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()

In [83]:
eeg_fooof_rest_noise_cont_matched = get_matched_legths(eeg_fooof_rest_noise_cont)


Subject: sub09, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 175 → 101 epochs (dropped last 74)
(128, 223, 101) (128, 223, 101)
Subject: sub10, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 96 → 71 epochs (dropped last 25)
(128, 223, 71) (128, 223, 71)
Subject: sub12, Paradigms: ['rest_init', 'pink_noise']
Truncated array1 from 140 → 116 epochs (dropped last 24)
(128, 223, 116) (128, 223, 116)


In [84]:
rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/rest_init_pink_cont_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]


_, rest_noise_fooof_bands_pvalues_aabs, _= get_channel_wise_fooof(freqs, eeg_fooof_rest_noise_cont_matched , bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(rest_noise_fooof_bands_pvalues_aabs)
# x, y= get_channel_wise_fooof(band_powers_rest_music["psds"], eeg_fooof_all_subs, bands)

plot_topomap_(fdr_corrected_p_vals, raw_info, pat_state, bands, rel_dif_fdr_fooof_fig_path, show_bands=True, 
              title="Rest vs. Pink Noise Cont", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub09
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub10
Paradigm 0 rest_init
Paradigm 1 pink_noise
sub12
Paradigm 0 rest_init
Paradigm 1 pink_noise
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


In [85]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/rest_init_pink_cont_abs.pdf"

_, eeg_fooof_bands_relative_rest_noise = get_fooof_bands(eeg_fooof_rest_noise_cont_matched, freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 


plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_rest_noise, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=True, title="(Pink Noise Cont. - Rest)/Rest", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm

Aperiodic

In [86]:
eeg_fooof_rest_noise_cont_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_noise_cont_matched)


plot_aperiodic_difference_all_subjects(
    eeg_fooof_rest_noise_cont_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/rest_init_pink_cont_abs_aperiodic.pdf",
    show_params=True,
    title="(PinkNoise Cont. - Rest)/Rest",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True,
)


Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Using Following Paradigm Names and in the following order:  ['rest_init', 'pink_noise']
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise


In [87]:
aperiodic_plots_path = plots_path + "frequency/" + "/rest_init_pink_cont_abs_aperiodic_fdr.pdf"
title_plot = "Rest vs. Pink Noise Cont"
plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_rest_noise_cont_aperiodic , raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels, show_params=True,
                                                               title=title_plot, 
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise
Paradigm 0:  rest_init
Paradigm 1:  pink_noise


### 4. Music Segs vs. Noise Segs

In [88]:
eeg_fooof_music_pink_segs = {}


eeg_fooof_music_pink_segs["sub07"] = {}
eeg_fooof_music_pink_segs["sub07"]["pink_segs"] = eeg_fooof_all_subs["sub07"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub07"]["music_segs"] = eeg_fooof_all_subs["sub07"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub09"] = {} 
eeg_fooof_music_pink_segs["sub09"]["pink_segs"] = eeg_fooof_all_subs["sub09"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub09"]["music_segs"] = eeg_fooof_all_subs["sub09"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub10"] = {} 
eeg_fooof_music_pink_segs["sub10"]["pink_segs"] = eeg_fooof_all_subs["sub10"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub10"]["music_segs"] = eeg_fooof_all_subs["sub10"]["music_segs"].copy()

eeg_fooof_music_pink_segs["sub12"] = {} 
eeg_fooof_music_pink_segs["sub12"]["pink_segs"] = eeg_fooof_all_subs["sub12"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub12"]["music_segs"] = eeg_fooof_all_subs["sub12"]["music_segs"].copy()


eeg_fooof_music_pink_segs_matched = get_matched_legths(eeg_fooof_music_pink_segs)

Subject: sub07, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 255 → 249 epochs (dropped last 6)
(128, 223, 249) (128, 223, 249)
Subject: sub09, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 188 → 185 epochs (dropped last 3)
(128, 223, 185) (128, 223, 185)
Subject: sub10, Paradigms: ['pink_segs', 'music_segs']
Truncated array2 from 136 → 69 epochs (dropped last 67)
(128, 223, 69) (128, 223, 69)
Subject: sub12, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 166 → 164 epochs (dropped last 2)
(128, 223, 164) (128, 223, 164)


In [89]:
rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/music_segs_pink_segs_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]


_, music_noise_fooof_bands_pvalues_abs, fooof_music_pink_segs = get_channel_wise_fooof(freqs, eeg_fooof_music_pink_segs_matched, bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(music_noise_fooof_bands_pvalues_abs)

# x, y= get_channel_wise_fooof(band_powers_rest_music["psds"], eeg_fooof_all_subs, bands)
plot_topomap_(fdr_corrected_p_vals , raw_info, pat_state, bands, rel_dif_fdr_fooof_fig_path, show_bands=1, 
              title="Music Segs vs. Pink Noise Segs", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub07
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub09
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub10
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub12
Paradigm 0 pink_segs
Paradigm 1 music_segs
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


In [90]:
mcs_minus_subjects = ["sub07", "sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_music_pink_segs,
    mcs_minus_subjects,
    condition1="music_segs",
    condition2="pink_segs",   # confirm this matches your actual music-condition key
    bands=bands_stats,
    abs_rel=0,
    n_boot=1000,
)

for band in bands_stats:
    print(f"{band}: {sig_mask[band].sum()} / 128 channels significant at group level")


latex_code_2 = latex_group_vs_individual_table(
    fdr_corrected_p_vals, mcs_minus_subjects, sig_mask, bands_stats,
    comparison_label="Segmented Music vs. Segmented Pink Noise",
    mode='append',
)
print(latex_code_2)




delta: 0 / 128 channels significant at group level
theta: 0 / 128 channels significant at group level
alpha: 0 / 128 channels significant at group level
beta: 1 / 128 channels significant at group level
gamma low: 0 / 128 channels significant at group level
\midrule

\multirow{5}{*}{\shortstack[l]{Segmented Music vs.\\Segmented Pink Noise}}
& $\delta$ & 0/128 & 11/128 & 0/128 ($\geq$2/4) & 0/128 \\
& $\theta$ & 0/128 & 0/128 & 0/128 ($\geq$2/4) & 0/128 \\
& $\alpha$ & 0/128 & 1/128 & 0/128 ($\geq$2/4) & 0/128 \\
& $\beta$ & 1/128 & 0/128 & 0/128 ($\geq$2/4) & 0/128 \\
& $\gamma$ & 0/128 & 0/128 & 0/128 ($\geq$2/4) & 0/128 \\




In [91]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/music_segs_pink_segs_abs.pdf"

_, eeg_fooof_bands_relative_music_noise = get_fooof_bands(eeg_fooof_music_pink_segs_matched, freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 

plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_music_noise, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=1, title="(Music Segs - Pink Noise Segs.)/Pink Noise Segs.", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm

Aperiodic

In [92]:
eeg_fooof_noise_music_aperiodic = get_fooof_aperiodic(eeg_fooof_music_pink_segs_matched)


plot_aperiodic_difference_all_subjects(
    eeg_fooof_noise_music_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/music_segs_pink_segs_abs_aperiodic.pdf",
    show_params=1,
    title="(Music Segs - Pink Noise Segs.)/Pink Noise Segs.",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True
)


Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Using Following Paradigm Names and in the following order:  ['pink_segs', 'music_segs']
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs


In [93]:
aperiodic_plots_path = plots_path + "frequency/" + "/music_segs_pink_segs_abs_aperiodic_fdr.pdf"
title_plot = "Music Segs vs. Pink Noise Segs"
n_channels = 128
plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_noise_music_aperiodic , raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels, show_params=1, 
                                                               title=title_plot, 
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs


### 5. Noise Cont vs. Reward

In [94]:
eeg_fooof_pink_noise_reward = {}

eeg_fooof_pink_noise_reward["sub09"] = {} 
eeg_fooof_pink_noise_reward["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub10"] = {} 
eeg_fooof_pink_noise_reward["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub12"] = {}
eeg_fooof_pink_noise_reward["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

eeg_fooof_pink_noise_reward_matched = get_matched_legths(eeg_fooof_pink_noise_reward)

Subject: sub09, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 134 → 101 epochs (dropped last 33)
(128, 223, 101) (128, 223, 101)
Subject: sub10, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 85 → 71 epochs (dropped last 14)
(128, 223, 71) (128, 223, 71)
Subject: sub12, Paradigms: ['pink_noise', 'reward']
Truncated array1 from 116 → 105 epochs (dropped last 11)
(128, 223, 105) (128, 223, 105)


In [95]:
rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/pink_noise_cont_music_cont_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]


_, music_noise_fooof_bands_pvalues_abs, fooof_pink_noise_reward = get_channel_wise_fooof(freqs, eeg_fooof_pink_noise_reward_matched, bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(music_noise_fooof_bands_pvalues_abs)

# x, y= get_channel_wise_fooof(band_powers_rest_music["psds"], eeg_fooof_all_subs, bands)
plot_topomap_(fdr_corrected_p_vals, raw_info, pat_state, bands_stats, rel_dif_fdr_fooof_fig_path, show_bands=True, 
              title="Pink Noise Cont. vs. Music Cont.", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub09
Paradigm 0 pink_noise
Paradigm 1 reward
sub10
Paradigm 0 pink_noise
Paradigm 1 reward
sub12
Paradigm 0 pink_noise
Paradigm 1 reward
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


d:\Taran\OneDrive - Radboud Universiteit\Documents\music_doc_study\analysis\..\helpers\topo_eeg.py:97: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(len(band_names), len(subjects), figsize=(4 * len(subjects), 3 * len(band_names)))


In [96]:
mcs_minus_subjects = ["sub09", "sub10", "sub12"]  # your actual MCS- subject IDs

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test(
    fooof_pink_noise_reward,
    mcs_minus_subjects,
    condition1="pink_noise",
    condition2="reward",   # confirm this matches your actual music-condition key
    bands=bands_stats,
    abs_rel=0,
    n_boot=1000,
)

for band in bands_stats:
    print(f"{band}: {sig_mask[band].sum()} / 128 channels significant at group level")


latex_code_2 = latex_group_vs_individual_table(
    fdr_corrected_p_vals, mcs_minus_subjects, sig_mask, bands_stats,
    comparison_label="Continuous Music vs. Continuous Pink Noise",
    mode='append',
)
print(latex_code_2)




c:\Users\specsdcn\anaconda3\envs\pdoc\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
c:\Users\specsdcn\anaconda3\envs\pdoc\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
c:\Users\specsdcn\anaconda3\envs\pdoc\Lib\site-packages\scipy\stats\_axis_nan_policy.py:430: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
c:\Users\specsdcn\anaconda3\envs\pdoc\Lib\site-packages\scipy\stats\_axis_nan_policy.py:

delta: 0 / 128 channels significant at group level
theta: 0 / 128 channels significant at group level
alpha: 0 / 128 channels significant at group level
beta: 0 / 128 channels significant at group level
gamma low: 0 / 128 channels significant at group level
\midrule

\multirow{5}{*}{\shortstack[l]{Continuous Music vs.\\Continuous Pink Noise}}
& $\delta$ & 0/128 & 0/128 & 0/128 ($\geq$2/3) & 0/128 \\
& $\theta$ & 0/128 & 0/128 & 0/128 ($\geq$2/3) & 0/128 \\
& $\alpha$ & 0/128 & 0/128 & 0/128 ($\geq$2/3) & 0/128 \\
& $\beta$ & 0/128 & 0/128 & 0/128 ($\geq$2/3) & 0/128 \\
& $\gamma$ & 0/128 & 3/128 & 0/128 ($\geq$2/3) & 0/128 \\




In [97]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/pink_noise_cont_music_cont_abs.pdf"

_, eeg_fooof_bands_relative_noise_reward = get_fooof_bands(eeg_fooof_pink_noise_reward_matched , freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 


plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_noise_reward, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=True, title="(Music Cont. - Pink Noise Cont.)/Pink Noise Cont.", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following

Aperiodic

In [98]:
eeg_fooof_noise_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_pink_noise_reward_matched)


plot_aperiodic_difference_all_subjects(
    eeg_fooof_noise_reward_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/pink_noise_cont_music_cont_abs_aperiodic.pdf",
    show_params=True,
    title="(Music Cont. - Pink Noise Cont.)/Pink Noise Cont.",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True
)


Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Using Following Paradigm Names and in the following order:  ['pink_noise', 'reward']
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward


In [99]:
aperiodic_plots_path = plots_path + "frequency/" + "/pink_noise_cont_music_cont_abs_aperiodic_fdr.pdf"
title_plot = "Pink Noise Cont. vs. Music Cont."

plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_noise_reward_aperiodic, raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels, show_params=True,
                                                               title=title_plot, 
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward


### 6. Rest Mid vs. Music Cont.

In [124]:
for sub in eeg_fooof_all_subs:
    print(sub)
    for parad in list(eeg_fooof_all_subs[sub].keys()):
        if "rest" in parad:
            print(parad)

sub01
rest_init
sub03
rest_init
sub04
rest_init
sub07
rest_end
rest_init
rest_mid
sub09
rest_end
rest_init
rest_mid
sub10
rest_init
rest_mid
sub12
rest_end
rest_init
rest_mid


In [125]:
eeg_fooof_rest_mid_reward = {}

eeg_fooof_rest_mid_reward["sub07"] = {}
eeg_fooof_rest_mid_reward["sub07"]["rest_mid"] = eeg_fooof_all_subs["sub07"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_mid_reward["sub09"] = {} 
eeg_fooof_rest_mid_reward["sub09"]["rest_mid"] = eeg_fooof_all_subs["sub09"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_mid_reward["sub10"] = {} 
eeg_fooof_rest_mid_reward["sub10"]["rest_mid"] = eeg_fooof_all_subs["sub10"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_mid_reward["sub12"] = {} 
eeg_fooof_rest_mid_reward["sub12"]["rest_mid"] = eeg_fooof_all_subs["sub12"]["rest_mid"].copy()
eeg_fooof_rest_mid_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [126]:
eeg_fooof_rest_mid_reward_matched = get_matched_legths(eeg_fooof_rest_mid_reward)


Subject: sub07, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 146 → 72 epochs (dropped last 74)
(128, 223, 72) (128, 223, 72)
Subject: sub09, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 134 → 87 epochs (dropped last 47)
(128, 223, 87) (128, 223, 87)
Subject: sub10, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 85 → 46 epochs (dropped last 39)
(128, 223, 46) (128, 223, 46)
Subject: sub12, Paradigms: ['rest_mid', 'reward']
Truncated array2 from 105 → 77 epochs (dropped last 28)
(128, 223, 77) (128, 223, 77)


In [127]:
sub_id = "sub10"

raw_info = eeg_data_all_subs[sub_id]["rest_init"].info
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]

rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/rest_mid_music_cont_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"

_, rest_mid_music_fooof_bands_pvalues_abs, fooof_rest_music = get_channel_wise_fooof(freqs, eeg_fooof_rest_mid_reward_matched, bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(rest_mid_music_fooof_bands_pvalues_abs)

# cmap_color = ListedColormap(["#D45E00", '#DAE8EC'])  # warm greige / muted amber


plot_topomap_(fdr_corrected_p_vals, raw_info, pat_state, bands_stats, rel_dif_fdr_fooof_fig_path, show_bands=True, 
              title="Rest Mid vs. Music Cont.", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub07
Paradigm 0 rest_mid
Paradigm 1 reward
sub09
Paradigm 0 rest_mid
Paradigm 1 reward
sub10
Paradigm 0 rest_mid
Paradigm 1 reward
sub12
Paradigm 0 rest_mid
Paradigm 1 reward
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


In [128]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/rest_mid_music_cont_abs.pdf"
_, eeg_fooof_bands_relative_rest_mid_music = get_fooof_bands(eeg_fooof_rest_mid_reward_matched, freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 

plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_rest_mid_music, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=True, title="(Music Cont. - Rest Mid)/Rest Mid", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mi

Aperiodic

In [129]:
eeg_fooof_rest_mid_music_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_mid_reward_matched)

plot_aperiodic_difference_all_subjects(
    eeg_fooof_rest_mid_music_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/rest_mid_music_cont_abs_aperiodic.pdf",
    show_params=True,
    title="(Music Cont. - Rest Mid)/Rest Mid",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True
)


Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_mid', 'reward']
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward


In [130]:
aperiodic_plots_path = plots_path + "frequency/" + "/rest_mid_music_cont_abs_aperiodic_fdr.pdf"
title_plot = "Music Cont. vs. Rest Mid"
n_channels = 128
plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_rest_mid_music_aperiodic, raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels,
                                                               title=title_plot, show_params=True,
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")

Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward
Paradigm 0:  rest_mid
Paradigm 1:  reward


### 7. Rest End vs. Music Cont.

In [131]:
eeg_fooof_rest_end_reward = {}

eeg_fooof_rest_end_reward["sub07"] = {}
eeg_fooof_rest_end_reward["sub07"]["rest_end"] = eeg_fooof_all_subs["sub07"]["rest_end"].copy()
eeg_fooof_rest_end_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_end_reward["sub09"] = {} 
eeg_fooof_rest_end_reward["sub09"]["rest_end"] = eeg_fooof_all_subs["sub09"]["rest_end"].copy()
eeg_fooof_rest_end_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_end_reward["sub12"] = {} 
eeg_fooof_rest_end_reward["sub12"]["rest_end"] = eeg_fooof_all_subs["sub12"]["rest_end"].copy()
eeg_fooof_rest_end_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [132]:
eeg_fooof_rest_end_reward_matched = get_matched_legths(eeg_fooof_rest_end_reward)


Subject: sub07, Paradigms: ['rest_end', 'reward']
Truncated array2 from 146 → 73 epochs (dropped last 73)
(128, 223, 73) (128, 223, 73)
Subject: sub09, Paradigms: ['rest_end', 'reward']
Truncated array2 from 134 → 87 epochs (dropped last 47)
(128, 223, 87) (128, 223, 87)
Subject: sub12, Paradigms: ['rest_end', 'reward']
Truncated array2 from 105 → 72 epochs (dropped last 33)
(128, 223, 72) (128, 223, 72)


In [133]:
sub_id = "sub10"

raw_info = eeg_data_all_subs[sub_id]["rest_init"].info
freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]   # fitted grid, identical for all subs/paradspsds = psds_all[sub_id]["rest_init"][0]

rel_dif_fdr_fooof_fig_path = plots_path + "frequency/" + "/rest_end_music_cont_abs_fdr.pdf"
c_bar_save_path = plots_path + "frequency/" + "/cbar.pdf"

_, rest_end_music_fooof_bands_pvalues_abs, fooof_rest_music = get_channel_wise_fooof(freqs, eeg_fooof_rest_end_reward_matched, bands_stats)
fdr_corrected_p_vals = get_fdr_p_values(rest_end_music_fooof_bands_pvalues_abs)

# cmap_color = ListedColormap(["#D45E00", '#DAE8EC'])  # warm greige / muted amber


plot_topomap_(fdr_corrected_p_vals, raw_info, pat_state, bands_stats, rel_dif_fdr_fooof_fig_path, show_bands=True, 
              title="Rest End vs. Music Cont.", cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)", save_path_cbar=c_bar_save_path)


sub07
Paradigm 0 rest_end
Paradigm 1 reward
sub09
Paradigm 0 rest_end
Paradigm 1 reward
sub12
Paradigm 0 rest_end
Paradigm 1 reward
['delta', 'theta', 'alpha', 'beta', 'gamma low']
{'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}


In [134]:
rel_dif_fooof_fig_path = plots_path + "frequency/" + "/rest_end_music_cont_abs.pdf"
_, eeg_fooof_bands_relative_rest_end_music = get_fooof_bands(eeg_fooof_rest_end_reward_matched, freqs, psds, bands_stats)
save_path_cbar = plots_path + "frequency/" 

plot_band_power_difference_all_subjects(eeg_fooof_bands_relative_rest_end_music, raw_info, pat_state, bands_stats, rel_dif_fooof_fig_path, show_all_cbars=True,
                                        show_bands=True, title="(Music Cont. - Rest End)/Rest End", cmap=soft_div, power_units_heading="Power Difference")



Band Display Names:  {'delta': '$\\mathbf{\\delta}$', 'theta': '$\\mathbf{\\theta}$', 'alpha': '$\\mathbf{\\alpha}$', 'beta': '$\\mathbf{\\beta}$', 'gamma low': '$\\mathbf{\\gamma}$'}
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_en

Aperiodic

In [135]:
eeg_fooof_rest_end_music_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_end_reward_matched )

plot_aperiodic_difference_all_subjects(
    eeg_fooof_rest_end_music_aperiodic,
    raw_info,
    pat_state,
    save_path=plots_path + "frequency/" + "/rest_end_music_cont_abs_aperiodic.pdf",
    show_params=True,
    title="(Music Cont. - Rest End)/Rest End",
    cmap=soft_div,
    value_units_heading="Power Difference",
    show_all_cbars=True
)


Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Using Following Paradigm Names and in the following order:  ['rest_end', 'reward']
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward


In [136]:
aperiodic_plots_path = plots_path + "frequency/" + "/rest_end_music_cont_abs_aperiodic_fdr.pdf"
title_plot = "Music Cont. vs. Rest End"
n_channels = 128
plot_band_power_difference_all_subjects_significance_aperiodic(eeg_fooof_rest_end_music_aperiodic, raw_info, 
                                                               pat_state, aperiodic_plots_path, n_channels,
                                                               title=title_plot, show_params=True,
                                                               cmap=cmap_color, power_units_heading="P-values: (0: <0.05 and 1: >=0.05)")

Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
Paradigm 0:  rest_end
Paradigm 1:  reward
